In [1]:
from datasets import load_dataset
raw_datasets = load_dataset("fancyzhx/ag_news")
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [2]:
#lets explore how a specific example looks:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'label': 2}

In [3]:
print(raw_train_dataset.features)

{'text': Value(dtype='string', id=None), 'label': ClassLabel(names=['World', 'Sports', 'Business', 'Sci/Tech'], id=None)}


In [4]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(batch):
    return tokenizer(
        batch["text"], truncation=True, padding=True, return_tensors="pt"
    )

tokenize_function(raw_train_dataset[:2])

{'input_ids': tensor([[  101,  2813,  2358,  1012,  6468, 15020,  2067,  2046,  1996,  2304,
          1006, 26665,  1007, 26665,  1011,  2460,  1011, 19041,  1010,  2813,
          2395,  1005,  1055,  1040, 11101,  2989,  1032,  2316,  1997, 11087,
          1011, 22330,  8713,  2015,  1010,  2024,  3773,  2665,  2153,  1012,
           102,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0],
        [  101, 18431,  2571,  3504,  2646,  3293, 13395,  1006, 26665,  1007,
         26665,  1011,  2797,  5211,  3813, 18431,  2571,  2177,  1010,  1032,
          2029,  2038,  1037,  5891,  2005,  2437,  2092,  1011, 22313,  1998,
          5681,  1032,  6801,  3248,  1999,  1996,  3639,  3068,  1010,  2038,
          5168,  2872,  1032,  2049, 29475,  2006,  2178,  2112,  1997,  1996,
          3006,  1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 

In [5]:
#we can use map() method to tokenize the whole dataset. This method applies a function to each element of the dataset in parallel:
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 7600
    })
})

In [7]:
import evaluate 

accuracy = evaluate.load("accuracy")
print(accuracy.description)
print(accuracy.compute(references=[0,1,0,1],predictions=[1,0,0,1]))


Accuracy is the proportion of correct predictions among the total number of cases processed. It can be computed with:
Accuracy = (TP + TN) / (TP + TN + FP + FN)
 Where:
TP: True positive
TN: True negative
FP: False positive
FN: False negative

{'accuracy': 0.5}


In [8]:
f1_score = evaluate.load("f1")

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    acc_result = accuracy.compute(references=labels, predictions=preds)
    acc = acc_result["accuracy"]

    f1_result = f1_score.compute(
        references = labels, predictions = preds, average="weighted"
    )

    f1 = f1_result["f1"]

    return {"accuracy": acc, "f1": f1}

In [10]:
import torch
from transformers import AutoModelForSequenceClassification

from genaibook.core import get_device

device = get_device()
num_labels = 4
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, num_labels= num_labels
).to(device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [19]:
from transformers import TrainingArguments

batch_size = 32
training_args = TrainingArguments(
    "classifier-chapter4",
    push_to_hub = False,
    num_train_epochs =2,
    eval_strategy = "epoch",
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size
)

In [20]:
from transformers import Trainer

shuffled_dataset = tokenized_datasets["train"].shuffle(seed=42)
small_split = shuffled_dataset.select(range(10000))

trainer = Trainer(
    model=model,
    args=training_args,
    compute_metrics = compute_metrics,
    train_dataset = small_split,
    eval_dataset = tokenized_datasets["test"],
    processing_class=tokenizer,
)

In [21]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.261211,0.911447,0.911374
2,0.300742,0.246468,0.920263,0.920235


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=626, training_loss=0.2767490959776857, metrics={'train_runtime': 217.9819, 'train_samples_per_second': 91.751, 'train_steps_per_second': 2.872, 'total_flos': 1875180164398464.0, 'train_loss': 0.2767490959776857, 'epoch': 2.0})

In [24]:
from transformers import pipeline

# 1. Eğittiğin modeli ve tokenizer'ı bir metin sınıflandırma pipeline'ına dönüştür
classifier = pipeline(
    "text-classification", 
    model=trainer.model, 
    tokenizer=tokenizer, 
    device=trainer.model.device
)

# 2. Veri setindeki orijinal etiket isimlerini (sınıfları) al
# Notebook'undaki ClassLabel: ['World', 'Sports', 'Business', 'Sci/Tech']
label_names = raw_train_dataset.features["label"].names

# 3. Test veri setinden kaç örnek görmek istiyorsun?
num_samples = 5
test_samples = tokenized_datasets["test"].select(range(num_samples))

# 4. Tahminleri yap ve ekrana yazdır
for i, sample in enumerate(test_samples):
    text = sample["text"]
    actual_label_id = sample["label"]
    actual_label_name = label_names[actual_label_id]
    
    # Model tahmini (Örn: {'label': 'LABEL_2', 'score': 0.98})
    prediction = classifier(text)[0]
    
    # Hugging face varsayılan olarak LABEL_0, LABEL_1 gibi döndürür. 
    # Buradaki rakamı ayıklayıp gerçek sınıf ismine dönüştürüyoruz:
    pred_label_id = int(prediction["label"].split("_")[-1])
    pred_label_name = label_names[pred_label_id]
    confidence = prediction["score"]
    
    print(f"--- Örnek {i+1} ---")
    print(f"Metin: {text[:150]}...") # Metnin ilk 150 karakteri
    print(f"Tahmin Edilen Sınıf: {pred_label_name} (%{confidence*100:.2f} güven)")
    print(f"Gerçek Sınıf: {actual_label_name}")
    print("-" * 40 + "\n")

--- Örnek 1 ---
Metin: Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Fed...
Tahmin Edilen Sınıf: Business (%98.10 güven)
Gerçek Sınıf: Business
----------------------------------------

--- Örnek 2 ---
Metin: The Race is On: Second Private Team Sets Launch Date for Human Spaceflight (SPACE.com) SPACE.com - TORONTO, Canada -- A second\team of rocketeers comp...
Tahmin Edilen Sınıf: Sci/Tech (%98.16 güven)
Gerçek Sınıf: Sci/Tech
----------------------------------------

--- Örnek 3 ---
Metin: Ky. Company Wins Grant to Study Peptides (AP) AP - A company founded by a chemistry researcher at the University of Louisville won a grant to develop ...
Tahmin Edilen Sınıf: Sci/Tech (%96.02 güven)
Gerçek Sınıf: Sci/Tech
----------------------------------------

--- Örnek 4 ---
Metin: Prediction Unit Helps Forecast Wildfires (AP) AP - It's barely dawn when Mike Fitzpatrick starts his shift with a bl